In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Self Attention

In [4]:
class SelfAttention(nn.Module):
    def __init__(self, embed_dim):
        """
        Standard Single-Head Self-Attention Module.
        Args:
            embed_dim (int): Dimensionality of the input tokens and hidden states.
        """
        super().__init__()
        # Linear projections for Queries, Keys, and Values
        self.to_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.to_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.to_v = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x):
        """
        Forward pass for self-attention.
        Args:
            x (torch.Tensor): Input sequence of shape [batch_size, seq_len, embed_dim]
        Returns:
            output (torch.Tensor): Attended context vectors of shape [batch_size, seq_len, embed_dim]
            attn (torch.Tensor): Attention weight matrix of shape [batch_size, seq_len, seq_len]
        """
        # x: [batch_size, seq_len, embed_dim]
        # Q, K, V: [batch_size, seq_len, embed_dim]
        Q = self.to_q(x)
        K = self.to_k(x)
        V = self.to_v(x)

        # Calculate scaling factor
        scale = math.sqrt(Q.size(-1))
        
        # K_t: [batch_size, embed_dim, seq_len]
        K_t = K.transpose(-2, -1)

        # Row alignment scores: [batch_size, seq_len, seq_len]
        scores = torch.matmul(Q, K_t) / scale 

        # Normalize so each row sums to 1.0
        # attn: [batch_size, seq_len, seq_len]
        attn = F.softmax(scores, dim=-1)

        # Compute the final weighted sum of values 
        # output: [batch_size, seq_len, embed_dim]
        output = torch.matmul(attn, V)
        return output, attn

In [38]:
embed_dim = 10
custom_attn = SelfAttention(embed_dim)
builtin_attn = nn.MultiheadAttention(embed_dim=embed_dim, 
                                     num_heads=1, 
                                     bias=False, 
                                     batch_first=True)

In [39]:
for name, param in builtin_attn.named_parameters():
    print(name, param.shape)

in_proj_weight torch.Size([30, 10])
out_proj.weight torch.Size([10, 10])


In [40]:
with torch.no_grad():
    builtin_attn.in_proj_weight.copy_(torch.cat([
        custom_attn.to_q.weight,
        custom_attn.to_k.weight,
        custom_attn.to_v.weight
    ], dim=0))
    builtin_attn.out_proj.weight.copy_(torch.eye(embed_dim))

    batch_size, seq_len = 4, 8
    x = torch.randn(batch_size, seq_len, embed_dim)

    custom_out, custom_attn_weights = custom_attn(x)
    print(f"{custom_out.shape} {custom_attn_weights.shape}")
    builtin_out, builtin_attn_weights = builtin_attn(x, x, x, need_weights=True)
    print(f"{builtin_out.shape} {builtin_attn_weights.shape}")
    
    out_match = torch.allclose(custom_out, builtin_out, atol=10e-6)
    attn_match = torch.allclose(custom_attn_weights, builtin_attn_weights, atol=10e-6)

    print(f"out_match: {out_match}")
    print(f"attn_match: {attn_match}")

torch.Size([4, 8, 10]) torch.Size([4, 8, 8])
torch.Size([4, 8, 10]) torch.Size([4, 8, 8])
out_match: True
attn_match: True


# MultiHeadAttention

In [51]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        """
        Standard MultiHeadAttention Module.
        Args:
            embed_dim (int): Dimensionality of the input tokens and hidden states.
            num_heads (int): Number of heads
        """
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        # Linear projections for Queries, Keys, and Values
        self.to_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.to_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.to_v = nn.Linear(embed_dim, embed_dim, bias=False)

        # Output projection
        self.to_out = nn.Linear(embed_dim, embed_dim, bias=False)
        

    def forward(self, x):
        """
        Forward pass for multi-head-attention.
        Args:
            x (torch.Tensor): Input sequence of shape [batch_size, seq_len, embed_dim]
        Returns:
            output (torch.Tensor): Attended context vectors of shape [batch_size, seq_len, embed_dim]
            attn (torch.Tensor): Attention weight matrix of shape [batch_size, seq_len, seq_len]
        """
        # x: [batch_size, seq_len, embed_dim]
        batch_size, seq_len, embed_dim = x.shape
        
        # Q, K, V: [batch_size, seq_len, embed_dim]
        Q = self.to_q(x)
        K = self.to_k(x)
        V = self.to_v(x)

        

        # Split to heads
        # Q, K, V: [batch_size, num_heads, seq_len, head_dim]
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
    

        # Calculate scaling factor
        scale = math.sqrt(self.head_dim)
        
        # K_t: [batch_size, num_heads, head_dim, seq_len]
        K_t = K.transpose(-2, -1)

        # Row alignment scores: [batch_size, num_heads, seq_len, seq_len]
        scores = torch.matmul(Q, K_t) / scale 

        # Normalize so each row sums to 1.0
        # attn: [batch_size, num_heads, seq_len, seq_len]
        attn = F.softmax(scores, dim=-1)

        # attended: [batch_size, num_heads, seq_len, head_dim]
        attended = torch.matmul(attn, V)

        # Concatenate heads
        # attended: [batch_size, seq_len, embed_dim]
        attended = attended.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)

        # Compute the final weighted sum of values 
        # output: [batch_size, seq_len, embed_dim]
        return self.to_out(attended), attn

In [52]:
embed_dim = 10
num_heads = 2
custom_attn = MultiHeadAttention(embed_dim, num_heads)
builtin_attn = nn.MultiheadAttention(embed_dim=embed_dim, 
                                     num_heads=num_heads, 
                                     bias=False,
                                     batch_first=True)

with torch.no_grad():
    builtin_attn.in_proj_weight.copy_(torch.cat([
        custom_attn.to_q.weight,
        custom_attn.to_k.weight,
        custom_attn.to_v.weight
    ], dim=0))
    builtin_attn.out_proj.weight.copy_(custom_attn.to_out.weight)

    batch_size, seq_len = 4, 8
    x = torch.randn(batch_size, seq_len, embed_dim)

    custom_out, custom_attn_weights = custom_attn(x)
    print(f"Custom: {custom_out.shape} {custom_attn_weights.shape}")
    builtin_out, builtin_attn_weights = builtin_attn(x, x, x, 
                                                     average_attn_weights=False,
                                                     need_weights=True)
    print(f"Builtin: {builtin_out.shape} {builtin_attn_weights.shape}")
    
    out_match = torch.allclose(custom_out, builtin_out, atol=10e-6)
    attn_match = torch.allclose(custom_attn_weights, builtin_attn_weights, atol=10e-6)

    print(f"out_match: {out_match}")
    print(f"attn_match: {attn_match}")

Custom: torch.Size([4, 8, 10]) torch.Size([4, 2, 8, 8])
Builtin: torch.Size([4, 8, 10]) torch.Size([4, 2, 8, 8])
out_match: True
attn_match: True
